In [8]:
import pdfplumber
import pandas as pd
import re

archivo_pdf = "Metals-ANNEXES-I-A-I-B-II-III-IV.pdf"
archivo_csv = "tablas_metales_completo.csv"

datos_procesados = []
anexo_actual = "Desconocido"

print("Extrayendo texto línea por línea para no omitir ninguna página...")

with pdfplumber.open(archivo_pdf) as pdf:
    for pagina in pdf.pages:
        # Extraer todo el texto de la página respetando el orden de lectura
        texto = pagina.extract_text()
        
        if not texto:
            continue
            
        for linea in texto.split('\n'):
            linea = linea.strip()
            
            # 1. Ignorar líneas vacías
            if not linea:
                continue
            
            # 2. Detectar si cambia el Anexo (Ej. Annex I-A)
            match_anexo = re.search(r'(?i)(Annex\s+[IVX]+(?:-[A-Z])?)', linea)
            if match_anexo:
                anexo_actual = match_anexo.group(1).upper().replace("ANNEX", "Annex")
                continue
                
            # 3. Ignorar encabezados de columnas y notas del documento
            encabezados_ignorados = ["Steel", "Description", "Aluminum", "Copper", 
                                     "Steel Derivatives", "Aluminum Derivatives", 
                                     "Copper Articles", "Derivatives"]
            
            if linea in encabezados_ignorados or linea.startswith("Note:") or linea.startswith("If a product") or linea.startswith("The following table"):
                continue
            
            # 4. Detectar Códigos HTS (al menos 4 números, pueden incluir puntos)
            # Ejemplos que atrapará: 7206, 7216.10.00, 8482.10.5004
            match_codigo = re.match(r'^(\d{4}[\d\.]*)(?:\s+(.*))?$', linea)
            
            if match_codigo:
                codigo = match_codigo.group(1).strip()
                # Si hay descripción en la misma línea, la guardamos; si no, queda en blanco temporalmente
                descripcion = match_codigo.group(2).strip() if match_codigo.group(2) else ""
                
                datos_procesados.append({
                    "Anexo": anexo_actual,
                    "Código (HTS)": codigo,
                    "Descripción": descripcion
                })
            else:
                # 5. Si la línea NO empieza con código, es la continuación de la descripción anterior
                # Evitamos que se pegue el número de página que suele estar solo al final
                if re.match(r'^\d+$', linea):
                    continue
                    
                if datos_procesados:
                    # Añadir un espacio y pegar el texto adicional
                    datos_procesados[-1]["Descripción"] += " " + linea

# Limpieza final para asegurar que no queden espacios dobles raros en las descripciones
for fila in datos_procesados:
    fila["Descripción"] = " ".join(fila["Descripción"].split())

# Convertir a DataFrame y guardar
df = pd.DataFrame(datos_procesados)
df.to_csv(archivo_csv, index=False, encoding='utf-8-sig')

print(f"¡Listo! Se extrajeron {len(df)} registros validados desde la página 1 en adelante.")
print(f"Revisa el nuevo archivo: '{archivo_csv}'")

Extrayendo texto línea por línea para no omitir ninguna página...
¡Listo! Se extrajeron 1198 registros validados desde la página 1 en adelante.
Revisa el nuevo archivo: 'tablas_metales_completo.csv'
